In [21]:
import polars as pl
import pathlib

In [2]:
data_dir = pathlib.Path().cwd() / "data"
files = list(data_dir.glob("*.csv"))

In [3]:
file_2021 = list(filter(lambda p: "2021" in p.name, files))[0]
file_2026 = list(filter(lambda p: "2026" in p.name, files))[0]

In [4]:
for f in (file_2021, file_2026):
    assert f.exists()

In [5]:
def read_file(path: pathlib.Path) -> pl.LazyFrame:
    return pl.scan_csv(source=path, separator=";").select(
        pl.col("gebiet-name", r"^D\d+$")
    )


q_old = read_file(file_2021)
q_new = read_file(file_2026)

In [6]:
party_map_new = {
    1: "CDU",
    2: "AfD",
    3: "SPD",
    4: "GRÜNE",
    5: "FDP",
    6: "DIE LINKE",
    7: "Volt",
    8: "PRO AUTO",
    9: "BLW",
    10: "Die PARTEI",
    11: "Die Gerechtigkeitspartei",
    12: "BSW",
    13: "FWG",
    14: "PdF",
    15: "FREIE WÄHLER",
}
party_map_old = {
    1: "CDU",
    2: "GRÜNE",
    3: "SPD",
    4: "AfD",
    5: "FDP",
    6: "Die Linke",
    7: "BLW",
    8: "FREIE WÄHLER",
    9: "ULW",
    10: "LKR",
    11: "Die PARTEI",
    12: "Volt",
    13: "BIG",
    14: "Pro Auto",
}

In [7]:
def normalize(d: dict[int, str]) -> dict[str, str]:
    return {f"D{k}": v.lower() for k, v in d.items()}


pmn = normalize(party_map_new)
pmo = normalize(party_map_old)

In [13]:
q_old_pr = q_old.rename(mapping=pmo)
q_new_pr = q_new.rename(mapping=pmn)

q = q_old_pr.join(
    q_new_pr, on="gebiet-name", how="left", validate="1:1", suffix=("_new")
)
df = q.collect()

In [14]:
obzs = df.select("gebiet-name").to_series().to_list()

In [34]:
old_parties = set(
    q_old_pr.select(pl.exclude(["year", "gebiet-name"])).collect_schema().names()
)
new_parties = set(
    q_new_pr.select(pl.exclude(["year", "gebiet-name"])).collect_schema().names()
)
parties = new_parties.intersection(old_parties)

In [38]:
def compare(party: str, df: pl.DataFrame = df) -> pl.DataFrame:
    q = df.select(pl.col("gebiet-name", f"^{party}.*$"))
    return q.with_columns(
        abs_change=pl.col(f"{party}_new") - pl.col(party),
        rel_change=(pl.col(f"{party}_new") - pl.col(party)) / pl.col(party) * 100,
    )


compare("afd")

gebiet-name,afd,afd_new,abs_change,rel_change
str,i64,i64,i64,f64
"""Mitte""",19515,39233,19718,101.040225
"""Nordost""",30888,62838,31950,103.438228
"""Südost""",27079,59904,32825,121.219395
"""Rheingauviertel, Hollerborn""",26081,53591,27510,105.479084
"""Klarenthal""",21636,47670,26034,120.327232
…,…,…,…,…
"""Medenbach""",6131,17640,11509,187.718154
"""Breckenheim""",5209,11236,6027,115.70359
"""Amöneburg""",2713,4909,2196,80.943605
